# Lab 14: What Survives the Trip from Paper to Portfolio?

*Financial Engineering in Practice*, Chapter 14.

**What you will explore:** how much of a backtest's profit remains once trading costs and capacity are counted.

You will take three famous long-short strategies (value, momentum and short-term reversal), estimate how much each trades, charge it for trading, and see which survive. Before each step, write down a prediction.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab14
from fintest.costs import simulate_universe

SOURCE = "french"   # "french" downloads real data; "sim" uses fake factor data to test the code
COST = 0.002        # one-way trading cost: 20 basis points
BORROW_FEE = 0.0025 # annual short-borrowing fee
OUT = Path("report")
OUT.mkdir(exist_ok=True)

df = lab14.load_factors(SOURCE)
SIM = df.attrs.get("simulated", False)
universe = simulate_universe(1000, 480, seed=2026)
print(df.index[0], "to", df.index[-1], "| simulated factors:", SIM)

## Part A: Gross performance

**Predict first:** rank the three strategies by Sharpe ratio before costs.

In [ ]:
a = lab14.part_a(df)
a.round(4)

## Part B: Turnover

**Predict first:** what share of the momentum portfolio do you think is replaced each month? Of the reversal portfolio?

In [ ]:
b = lab14.part_b(universe)
lab14.fig_turnover(b, OUT)
display(Image(OUT / "fig14_3_turnover.png"))
b.round(3)

## Part C: Costs

**Predict first:** what one-way cost would wipe out each strategy's average return? Will the ranking from Part A survive?

In [ ]:
c = lab14.part_c(df, lab14.PUBLISHED_TURNOVER, COST, BORROW_FEE)
lab14.fig_net_vs_cost(c, OUT, SIM, COST, f"{df.index[0]} to {df.index[-1]}")
lab14.fig_cumulative(df, lab14.PUBLISHED_TURNOVER, COST, BORROW_FEE, OUT, SIM)
display(Image(OUT / "fig14_1_net_vs_cost.png"))
display(Image(OUT / "fig14_2_cumulative.png"))
c["table"][["strategy", "gross_mean", "net_mean", "gross_sharpe", "net_sharpe", "break_even_one_way",
            "rank_gross", "rank_net"]].round(4)

In [ ]:
lab14.cost_sensitivity(df, lab14.PUBLISHED_TURNOVER, BORROW_FEE).round(2)

## Part D: Capacity

**Predict first:** how large could a momentum fund grow before trading costs ate half its return? $100 million? $10 billion? $100 billion?

In [ ]:
micro = lab14.market_microstructure(universe, 50e12)
gross = dict(zip(a["key"], a["mean_monthly"] - BORROW_FEE / 12))
d = lab14.part_d(universe, gross, lab14.PUBLISHED_TURNOVER, micro, y=1.0, days=5)
lab14.fig_capacity(d, OUT, SIM, {"y": 1.0, "days": 5, "total_cap": 50e12})
display(Image(OUT / "fig14_4_capacity.png"))
{k: (lab14._money(d[k]["half_net_aum"]), lab14._money(d[k]["zero_net_aum"])) for k in lab14.STRATEGIES}

Try it: re-run Part D with `y=0.5` (cheaper impact) and with `days=20` (slower trading). Which assumption moves capacity most?

## Part E: Mitigation with known truth

**Predict first:** if momentum holds stocks until they fall out of the top 50% instead of the top 30%, how much turnover will it save, and how much expected return will it give up?

In [ ]:
e = lab14.part_e(universe, COST)
lab14.fig_banding(e, OUT, COST)
display(Image(OUT / "fig14_5_banding.png"))
e.drop(columns=["key"]).round(4)

## Questions to think about

1. Which strategies are profitable to trade at realistic costs, and which only on paper?
2. How does the ranking change after costs, and why?
3. Which assumption in Part D matters most, and how would you check it?
4. Is it fair to compare a cost-aware strategy (Part E) with the published paper version? What would you need to report?